# Importation des bibliothèques

In [38]:
import os
import sys
import findspark
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, when, regexp_replace, trim
from delta import configure_spark_with_delta_pip

In [39]:
# Configuration des variables d'environnement pour Java, Spark et Hadoop
os.environ["JAVA_HOME"] = r"C:\Users\ahoun\AppData\Local\Programs\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
os.environ["SPARK_HOME"] = r"C:\spark\spark-3.5.6-bin-hadoop3"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

findspark.init()

# Constantes

In [40]:
# chemin pour accéder aux données
FILE_PATH = '../src/data/raw/donnees_immobilieres.parquet'

# chemin pour sauvegarder les données nettoyées
OUTPUT_PATH = '../src/data/clean/donnees_immobilieres_cleaned.delta'

# Application

In [41]:
# Init SparkSession avec Delta Lake
builder = (
    SparkSession.builder
    .appName("RealStateProcessing")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "4g") 
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", "C:/tmp/spark-warehouse")
)

# On configure et on démarre
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [42]:
# chargement des données
df_raw = spark.read.parquet(FILE_PATH)

In [43]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df_raw.count(), len(df_raw.columns))

(10000, 13)

In [44]:
# Affichage du schéma
df_raw.printSchema()

root
 |-- region: string (nullable = true)
 |-- ue: string (nullable = true)
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- fonction: string (nullable = true)
 |-- designation_batiment_terrain: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- pays: string (nullable = true)
 |-- ministere: string (nullable = true)
 |-- date_inventaire: string (nullable = true)



In [45]:
# Affichage des 5 premières lignes
df_raw.show(5)

+------------------+--------------------+--------------------+--------------------+--------------------+----------------------------+----+--------------------+-----------+-----------------+------+--------------------+---------------+
|            region|                  ue|                  id|                type|            fonction|designation_batiment_terrain|dept|             adresse|code_postal|            ville|  pays|           ministere|date_inventaire|
+------------------+--------------------+--------------------+--------------------+--------------------+----------------------------+----+--------------------+-----------+-----------------+------+--------------------+---------------+
|             CORSE|a19e7cbea4c2dae3c...|936465ba5b332c604...|biens du ministèr...|biens du ministèr...|        biens du ministèr...|  2B|                NULL|       NULL|             NULL|France|Intérieur, Outre-...|        2014-03|
|NORD-PAS DE CALAIS|c1fb47969da8dd1a5...|44b7ec415ee6f810d...|bi

In [46]:
# suppression des doublons suivant l'id
df = df_raw.dropDuplicates(['id'])

In [47]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df.count(), len(df.columns))

(9041, 13)

In [48]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
df.groupBy("id").count().filter(col("count") > 1).count()

0

In [49]:
# filtrage pour ne garder que les biens en France
df = df.filter(col("pays") == "France")

In [50]:
# affichage des pays distincts
df.select("pays").distinct().show()

+------+
|  pays|
+------+
|France|
+------+



In [51]:
# conversion de la colonne "date_invnetaire" en type date
df = df.withColumn("date_inventaire", to_date(col("date_inventaire"), "yyyy-MM"))

In [52]:
# Affichage du schéma
df.printSchema()

root
 |-- region: string (nullable = true)
 |-- ue: string (nullable = true)
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- fonction: string (nullable = true)
 |-- designation_batiment_terrain: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- pays: string (nullable = true)
 |-- ministere: string (nullable = true)
 |-- date_inventaire: date (nullable = true)



In [53]:
# mise en minuscule de la colonne "ville" pour uniformiser les données
df = df.withColumn("ville", lower(col("ville")))

In [54]:
# affichage des villes distinctes
df.select("ville").distinct().show()

+--------------------+
|               ville|
+--------------------+
|          la trinité|
|               cergy|
|     gisy-les-nobles|
|   varennes-le-grand|
|    condé-sur-sarthe|
|mauzac-et-grand-c...|
|              coings|
|        la ricamarie|
|                pacé|
|     choloy-ménillot|
|              lescun|
|              mauzac|
|                reau|
|               royan|
|      pointe-à-pitre|
|            dzaoudzi|
|              bailly|
|   montigny-lès-metz|
|              lognes|
|                belz|
+--------------------+
only showing top 20 rows



In [55]:
# mise en minuscule de la colonne "region" pour uniformiser les données
df = df.withColumn("region", lower(col("region")))

In [56]:
# affichage des régions distinctes
df.select("region").distinct().show()

+--------------------+
|              region|
+--------------------+
|            bretagne|
|       ile de france|
|               c.o.m|
|            limousin|
|  nord-pas de calais|
|       franche comte|
|     basse normandie|
|            auvergne|
|               d.o.m|
|           aquitaine|
|languedoc-roussillon|
|            lorraine|
|         rhone alpes|
|      midi-pyrennees|
|              centre|
|       pays de loire|
|           bourgogne|
|  champagne-ardennes|
|                paca|
|            picardie|
+--------------------+
only showing top 20 rows



In [57]:
# anonymisation des données sensibles
df = df.withColumn("ville",when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("ville")))

In [58]:
# affichage des villes distinctes
df.select("ville").distinct().show()

+--------------------+
|               ville|
+--------------------+
|          la trinité|
|               cergy|
|     gisy-les-nobles|
|   varennes-le-grand|
|    condé-sur-sarthe|
|mauzac-et-grand-c...|
|              coings|
|        la ricamarie|
|                pacé|
|     choloy-ménillot|
|              lescun|
|              mauzac|
|                reau|
|               royan|
|      pointe-à-pitre|
|            dzaoudzi|
|              bailly|
|   montigny-lès-metz|
|              lognes|
|                belz|
+--------------------+
only showing top 20 rows



In [59]:
# anonymisation des données sensibles
df = df.withColumn("code_postal", when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("code_postal")))

In [60]:
# affichage des codes postaux distincts
df.select("code_postal").distinct().show()

+-----------+
|code_postal|
+-----------+
|      75007|
|      42370|
|      59370|
|      59169|
|      77930|
|      09120|
|      16250|
|      91910|
|      86180|
|      50800|
|      66000|
|      27130|
|      31330|
|      57655|
|      59680|
|      91120|
|      28500|
|      22130|
|      57500|
|      57180|
+-----------+
only showing top 20 rows



In [61]:
# anonymisation des données sensibles
df = df.withColumn("adresse", when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("adresse")))

In [62]:
# affichage des adresses distinctes
df.select("adresse").distinct().show()

+--------------------+
|             adresse|
+--------------------+
|  11 Rue ELIE VERNET|
|     21 Route D AUCH|
|R DU MARECHAL LEC...|
|        29 R DELILLE|
|Lieu-dit ILET DU ...|
|     Rue DE L EGLISE|
|159  AV JACQUES D...|
|LD LE MONT FLOLENTIN|
|    LIEU-DIT MONTEAU|
|LD LA TOUR DE CHA...|
|      13 RUE GALILEE|
|  LD LA MARE POISSON|
|9  AV DOCTEUR MAU...|
|9080 ALL DE LA RO...|
|85/87  RTE DE CALAIS|
|      LD BEAU RIVAGE|
|  LD LA GRANDE GREVE|
|  LD LES GRIMPETEAUX|
|         LD KERMARIO|
|RTE DE COURCOURONNES|
+--------------------+
only showing top 20 rows



In [64]:
df.select("ministere").distinct().count()

42

In [63]:
# nettoyage de la colonne "ministere" : suppression des espaces, mise en minuscule et suppression des ponctuations finales
df = df.withColumn("ministere", regexp_replace(trim(lower(col("ministere"))), r'[.,;]+$', '')
)

In [65]:
# uniformisation des données du ministère
df = df.withColumn("ministere", when(col("ministere").like("%intérieur%"), "ministère de l'Intérieur")
    .when(col("ministere").like("%éducation%") | col("ministere").like("%education%") | 
        col("ministere").like("%enseignement sup%"), "ministère de l'Éducation Nationale")
    .when(col("ministere").like("%justice%"), "ministère de la Justice")
    .when(col("ministere").like("%santé%"), "ministère de la Santé")
    .when(col("ministere").like("%ecologie%") | col("ministere").like("%écologie%") | 
          col("ministere").like("%environnement%") | col("ministere").like("%aménagement%"), 
        "ministère de l'Écologie")
    .when(col("ministere").like("%budget%") | col("ministere").like("%comptes publics%") | 
          col("ministere").like("%fonction publique%"), "ministère du Budget")
    .when(col("ministere").like("%affaires étrangères%") | col("ministere").like("%affaires etrangeres%") | 
          col("ministere").like("%européennes%"), "ministère des Affaires Étrangères")
    .when(col("ministere").like("%economie%") | col("ministere").like("%économie%") | 
          col("ministere").like("%finances%") | col("ministere").like("%industrie%"), "ministère de l'Économie et des Finances")
    .when(col("ministere").like("%travail%") | col("ministere").like("%relations soc%") | 
          col("ministere").like("%solidarité%"), "ministère du Travail")
    .when(col("ministere").like("%culture%"), "ministère de la Culture")
    .when(col("ministere").like("%agriculture%") | col("ministere").like("%pêche%"), "ministère de l'Agriculture")
    .when(col("ministere").like("%premier ministre%") | 
          col("ministere").like("%services du premier%"), "services du Premier Ministre")
    .when(col("ministere").like("%aviation%"), "aviation civile")
    .when(col("ministere").like("%sénat%") | col("ministere").like("%senat%"), "Sénat")
    .when(col("ministere").like("%pouvoirs publics%"), "pouvoirs publics")
    .when(col("ministere").like("%multi%occupant%") | col("ministere").like("%multioccup%"), "multi-occupants")
    .when(col("ministere").like("%france domaine%") | col("ministere").like("%domaine%"), "France Domaine")
    .when(col("ministere").like("%conseil d%état%") | col("ministere").like("%conseil d'etat%"), "Conseil d'État")
    .when(col("ministere").like("%assemblée nationale%"), "Assemblée Nationale").otherwise(col("ministere"))
)

In [66]:
df.select("ministere").distinct().count()

18

In [67]:
# affichage des ministères distincts
df.select("ministere").distinct().show()

+--------------------+
|           ministere|
+--------------------+
|ministère de l'Éc...|
| Assemblée Nationale|
|ministère des Aff...|
|ministère de l'Éd...|
|      Conseil d'État|
|ministère du Travail|
|     multi-occupants|
|services du Premi...|
|               Sénat|
|ministère de la J...|
|     aviation civile|
|ministère de l'In...|
|      France Domaine|
| ministère du Budget|
|    pouvoirs publics|
|ministère de la S...|
|ministère de la C...|
|ministère de l'Éc...|
+--------------------+



In [68]:
# Affichage du schéma
df.printSchema()

root
 |-- region: string (nullable = true)
 |-- ue: string (nullable = true)
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- fonction: string (nullable = true)
 |-- designation_batiment_terrain: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- pays: string (nullable = true)
 |-- ministere: string (nullable = true)
 |-- date_inventaire: date (nullable = true)



In [69]:
# Affichage des 5 premières lignes
df.show(5)

+-------------+--------------------+--------------------+--------------------+--------------------+----------------------------+----+-------+-----------+-----+------+--------------------+---------------+
|       region|                  ue|                  id|                type|            fonction|designation_batiment_terrain|dept|adresse|code_postal|ville|  pays|           ministere|date_inventaire|
+-------------+--------------------+--------------------+--------------------+--------------------+----------------------------+----+-------+-----------+-----+------+--------------------+---------------+
|ile de france|c0daef7745cdaf58f...|000d84e011f2b9e7e...|biens du ministèr...|biens du ministèr...|        biens du ministèr...|  93|   NULL|       NULL| NULL|France|ministère de l'In...|     2014-03-01|
|     limousin|1067ef9e24a6dcadc...|0020d6985224f09cd...|biens du ministèr...|biens du ministèr...|        biens du ministèr...|  23|   NULL|       NULL| NULL|France|ministère de l'In.

In [70]:
# Informations générales sur le DataFrame 
df_raw.describe().show()

+-------+-----------+--------------------+--------------------+----------------+-------------------+----------------------------+-----------------+--------------------+------------------+-----------+-----------+-------------------+---------------+
|summary|     region|                  ue|                  id|            type|           fonction|designation_batiment_terrain|             dept|             adresse|       code_postal|      ville|       pays|          ministere|date_inventaire|
+-------+-----------+--------------------+--------------------+----------------+-------------------+----------------------------+-----------------+--------------------+------------------+-----------+-----------+-------------------+---------------+
|  count|      10000|                9703|               10000|           10000|              10000|                       10000|             9998|                6712|              6985|       6972|      10000|              10000|          10000|
|   mean

In [71]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df.count(), len(df.columns))

(8883, 13)

In [72]:
# Création du dossier si il n'existe pas
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

In [73]:
# Sauvegarde des données nettoyées au format delta
df.write.format("delta").mode("overwrite").save(OUTPUT_PATH)